# LLM Checkpoint Manifest：Shard 完整性、顺序与兼容加载

**面试问题：分布式训练 Checkpoint 有很多 Shard，恢复时如何避免顺序、缺失和配置不兼容？**

## 回答主线

1. 不能依赖目录 glob 顺序猜测参数布局，Manifest 必须显式列出每个张量的 shape、dtype、切分维、范围和 shard。
2. 每个 shard 需要字节数与强哈希，加载前先验证完整性再分配大张量。
3. 重建顺序来自 tensor range，而不是文件名的字典序。
4. 模型结构、Tokenizer、并行策略和优化器版本也属于恢复合同。
5. 损坏、重复覆盖、范围空洞和 world size 变化都应给出可操作错误。
6. 生产系统还需要原子完成标记、远端对象存储和弹性重分片。

## 真实案例

一个 `lm_head.weight` 教学张量 shape=(6,4) 被三张卡按行切成 3 个 shard，文件名故意为 shard-1、shard-2、shard-10，字典序会错。我们用 NumPy 原始字节构造 Manifest、验证 SHA-256、按 row range 重建，并篡改一个 shard 及模型 hidden_size。案例使用仓库内生成的离线脱敏小数据，输出用于学习机制，不代表生产性能。

### 输入预览：张量、三份 Shard 与错误文件名顺序

In [1]:
import hashlib  # 导入哈希函数以验证 shard 字节完整性。
import json  # 导入 JSON 以展示稳定 Manifest。
import numpy as np  # 导入数组运算以切分和重建教学张量。

tensor = np.arange(24, dtype=np.float32).reshape(6, 4)  # 构造可读的六行四列模型权重。
shard_specs = [  # 故意使用不按数值字典序排列的文件名。
    {"file": "shard-1.bin", "row_start": 0, "row_end": 2},  # 保存前两行。
    {"file": "shard-2.bin", "row_start": 2, "row_end": 4},  # 保存中两行。
    {"file": "shard-10.bin", "row_start": 4, "row_end": 6},  # 保存后两行。
]  # 完成切分规范。
blob_store = {}  # 用内存对象存储模拟 checkpoint 文件。
for spec in shard_specs:  # 逐规范序列化张量分片。
    shard = tensor[spec["row_start"]:spec["row_end"]]  # 按行范围切片。
    blob_store[spec["file"]] = shard.tobytes(order="C")  # 保存连续原始字节。
print("原张量：\n", tensor)  # 展示重建目标。
print("文件名字典序：", sorted(blob_store))  # 展示 shard-10 会排在 shard-2 前。
for spec in shard_specs:  # 逐 shard 展示行范围和字节数。
    print(f"{spec['file']:<12} rows=[{spec['row_start']},{spec['row_end']}) bytes={len(blob_store[spec['file']])}")  # 展示物理布局。

原张量：
 [[ 0.  1.  2.  3.]
 [ 4.  5.  6.  7.]
 [ 8.  9. 10. 11.]
 [12. 13. 14. 15.]
 [16. 17. 18. 19.]
 [20. 21. 22. 23.]]
文件名字典序： ['shard-1.bin', 'shard-10.bin', 'shard-2.bin']
shard-1.bin  rows=[0,2) bytes=32
shard-2.bin  rows=[2,4) bytes=32
shard-10.bin rows=[4,6) bytes=32


## Baseline 基线：按文件名字典序直接 Concatenate

In [2]:
def decode_blob(blob, columns=4):  # 把 FP32 原始字节恢复为二维 shard。
    return np.frombuffer(blob, dtype=np.float32).reshape(-1, columns)  # 按已知列数解析字节。

glob_order = sorted(blob_store)  # 模拟目录 glob 返回字典序文件名。
naive_rebuilt = np.concatenate([decode_blob(blob_store[file]) for file in glob_order], axis=0)  # 按错误文件顺序拼接。
naive_equal = np.array_equal(naive_rebuilt, tensor)  # 比较重建张量与训练权重。
print("朴素重建：\n", naive_rebuilt)  # 展示最后四行发生顺序交换。
print("与原张量一致：", naive_equal)  # 暴露不崩溃但参数语义错误。
print("错误行对照：", [(index, tensor[index].tolist(), naive_rebuilt[index].tolist()) for index in range(6) if not np.array_equal(tensor[index], naive_rebuilt[index])])  # 展示具体错位行。

朴素重建：
 [[ 0.  1.  2.  3.]
 [ 4.  5.  6.  7.]
 [16. 17. 18. 19.]
 [20. 21. 22. 23.]
 [ 8.  9. 10. 11.]
 [12. 13. 14. 15.]]
与原张量一致： False
错误行对照： [(2, [8.0, 9.0, 10.0, 11.0], [16.0, 17.0, 18.0, 19.0]), (3, [12.0, 13.0, 14.0, 15.0], [20.0, 21.0, 22.0, 23.0]), (4, [16.0, 17.0, 18.0, 19.0], [8.0, 9.0, 10.0, 11.0]), (5, [20.0, 21.0, 22.0, 23.0], [12.0, 13.0, 14.0, 15.0])]


### 核心实现：显式范围、字节数和 SHA Manifest

In [3]:
manifest = {  # 构造完整 checkpoint 合同。
    "format_version": 1,  # 标记 Manifest 格式版本。
    "model": {"architecture": "TinyLM", "hidden_size": 4, "vocab_size": 6},  # 绑定模型结构。
    "tokenizer_hash": "tok-45aa",  # 绑定 Token ID 语义。
    "training_world_size": 3,  # 记录保存时并行规模。
    "tensors": {"lm_head.weight": {"shape": [6, 4], "dtype": "float32", "shard_dim": 0, "shards": []}},  # 描述目标张量与切分维。
}  # 完成 Manifest 骨架。
for spec in shard_specs:  # 逐 shard 补齐完整性字段。
    blob = blob_store[spec["file"]]  # 读取原始字节。
    manifest["tensors"]["lm_head.weight"]["shards"].append({**spec, "bytes": len(blob), "sha256": hashlib.sha256(blob).hexdigest()})  # 保存范围、大小和哈希。
print(json.dumps(manifest, ensure_ascii=False, indent=2))  # 展示可供加载器消费的真实 Manifest。

def validate_manifest(checkpoint_manifest, blobs, runtime_model):  # 在加载前验证结构、范围和完整性。
    errors = []  # 收集全部可操作错误。
    if checkpoint_manifest["model"] != runtime_model:  # 检查模型结构完全一致。
        errors.append("model-config-mismatch")  # 保存配置错误。
    tensor_record = checkpoint_manifest["tensors"]["lm_head.weight"]  # 读取目标张量记录。
    expected_row = 0  # 跟踪连续行范围起点。
    ordered = sorted(tensor_record["shards"], key=lambda item: item["row_start"])  # 按逻辑范围而非文件名排序。
    for item in ordered:  # 逐 shard 验证范围和字节。
        if item["row_start"] != expected_row:  # 检查范围是否有空洞或重叠。
            errors.append(f"row-range-gap:{expected_row}->{item['row_start']}")  # 保存范围错误。
        expected_row = item["row_end"]  # 推进期望结束行。
        blob = blobs.get(item["file"])  # 读取对象存储中的 shard。
        if blob is None:  # 文件缺失时无法继续哈希。
            errors.append(f"missing:{item['file']}")  # 保存缺失文件。
            continue  # 跳过当前 shard 后续检查。
        if len(blob) != item["bytes"]:  # 检查实际字节数。
            errors.append(f"size:{item['file']}")  # 保存截断或追加错误。
        if hashlib.sha256(blob).hexdigest() != item["sha256"]:  # 检查内容哈希。
            errors.append(f"hash:{item['file']}")  # 保存静默损坏错误。
    if expected_row != tensor_record["shape"][0]:  # 检查范围覆盖目标全部行。
        errors.append(f"row-range-end:{expected_row}")  # 保存尾部空洞。
    return len(errors) == 0, errors, ordered  # 返回结论、错误和正确顺序。

valid, errors, ordered_specs = validate_manifest(manifest, blob_store, manifest["model"])  # 验证原始 checkpoint。
print("验证结果：", valid, errors, "逻辑顺序=", [item["file"] for item in ordered_specs])  # 展示 Manifest 修复字典序。

{
  "format_version": 1,
  "model": {
    "architecture": "TinyLM",
    "hidden_size": 4,
    "vocab_size": 6
  },
  "tokenizer_hash": "tok-45aa",
  "training_world_size": 3,
  "tensors": {
    "lm_head.weight": {
      "shape": [
        6,
        4
      ],
      "dtype": "float32",
      "shard_dim": 0,
      "shards": [
        {
          "file": "shard-1.bin",
          "row_start": 0,
          "row_end": 2,
          "bytes": 32,
          "sha256": "0571cfe42be5c7b95de9afc7c7ba1286fb7a2ef10a9035f8d6b87d21a3bc8387"
        },
        {
          "file": "shard-2.bin",
          "row_start": 2,
          "row_end": 4,
          "bytes": 32,
          "sha256": "45701da4b3c9bd087207d34d5ec61fe12c9ed32fe2f5b509dd7ea08799a9a8e8"
        },
        {
          "file": "shard-10.bin",
          "row_start": 4,
          "row_end": 6,
          "bytes": 32,
          "sha256": "9b06c638288d1c3792a12fe41901ae67d25668183dfebbe1b248bf2319d06f4d"
        }
      ]
    }
  }
}
验证结果： True 

## 结果解读：先验证再按 Row Range 重建

In [4]:
def load_tensor(checkpoint_manifest, blobs, runtime_model):  # 安全加载并重建教学张量。
    valid, errors, ordered = validate_manifest(checkpoint_manifest, blobs, runtime_model)  # 在分配大张量前运行全部门禁。
    if not valid:  # 任一结构或完整性错误都拒绝加载。
        return None, errors  # 返回错误而不产生部分参数。
    tensor_record = checkpoint_manifest["tensors"]["lm_head.weight"]  # 读取 dtype 和 shape。
    parts = [np.frombuffer(blobs[item["file"]], dtype=np.dtype(tensor_record["dtype"])).reshape(item["row_end"] - item["row_start"], tensor_record["shape"][1]) for item in ordered]  # 按逻辑范围解码所有 shard。
    rebuilt = np.concatenate(parts, axis=tensor_record["shard_dim"])  # 沿显式切分维拼接。
    return rebuilt, []  # 返回完整张量。

safe_rebuilt, load_errors = load_tensor(manifest, blob_store, manifest["model"])  # 加载合法 checkpoint。
print("方案             文件顺序                         与原张量一致")  # 输出同口径对照表头。
print(f"Glob concat      {glob_order}  {naive_equal}")  # 展示文件名字典序失败。
print(f"Manifest ranges  {[item['file'] for item in ordered_specs]}  {np.array_equal(safe_rebuilt, tensor)}")  # 展示逻辑范围正确。
print("安全重建张量：\n", safe_rebuilt)  # 展示逐值恢复结果。
print("解读：文件名只是对象标识，真正的拼接顺序和形状都来自签名 Manifest。")  # 解释核心原则。

方案             文件顺序                         与原张量一致
Glob concat      ['shard-1.bin', 'shard-10.bin', 'shard-2.bin']  False
Manifest ranges  ['shard-1.bin', 'shard-2.bin', 'shard-10.bin']  True
安全重建张量：
 [[ 0.  1.  2.  3.]
 [ 4.  5.  6.  7.]
 [ 8.  9. 10. 11.]
 [12. 13. 14. 15.]
 [16. 17. 18. 19.]
 [20. 21. 22. 23.]]
解读：文件名只是对象标识，真正的拼接顺序和形状都来自签名 Manifest。


## 失败案例：单字节损坏与 Hidden Size 不兼容

In [5]:
corrupted_store = dict(blob_store)  # 复制对象存储以注入损坏。
damaged = bytearray(corrupted_store["shard-2.bin"])  # 将中间 shard 转为可修改字节数组。
damaged[3] ^= 0x01  # 翻转一个字节模拟传输或磁盘静默损坏。
corrupted_store["shard-2.bin"] = bytes(damaged)  # 写回损坏内容。
corrupt_tensor, corrupt_errors = load_tensor(manifest, corrupted_store, manifest["model"])  # 尝试加载损坏 checkpoint。
wrong_runtime = {"architecture": "TinyLM", "hidden_size": 8, "vocab_size": 6}  # 构造不同隐藏宽度的运行模型。
wrong_tensor, config_errors = load_tensor(manifest, blob_store, wrong_runtime)  # 尝试把四列权重加载到八维模型。
print(f"字节损坏：tensor={corrupt_tensor} errors={corrupt_errors}")  # 展示哈希在解码前阻断。
print(f"配置不兼容：tensor={wrong_tensor} errors={config_errors}")  # 展示结构门禁阻断。
print("修正策略：错误不可通过 reshape 或 strict=False 静默绕过；应重新下载损坏 shard，或运行显式且可验证的转换工具。")  # 总结故障处理。

字节损坏：tensor=None errors=['hash:shard-2.bin']
配置不兼容：tensor=None errors=['model-config-mismatch']
修正策略：错误不可通过 reshape 或 strict=False 静默绕过；应重新下载损坏 shard，或运行显式且可验证的转换工具。


### 生产边界与恢复事件

In [6]:
restore_event = {"checkpoint": "step-4200", "manifest_version": manifest["format_version"], "training_world_size": manifest["training_world_size"], "runtime_world_size": 2, "tensor": "lm_head.weight", "status": "validated-and-reshard-required"}  # 构造弹性恢复事件。
print("恢复事件：", restore_event)  # 展示 world size 变化需要重分片而非拒绝整个逻辑张量。
print("生产替换点：真实 Checkpoint 还需 safetensors、优化器/RNG、原子 COMPLETE 标记、对象存储 ETag、分布式并行布局、流式重分片和磁盘容量门禁。")  # 明确内存字节边界。

恢复事件： {'checkpoint': 'step-4200', 'manifest_version': 1, 'training_world_size': 3, 'runtime_world_size': 2, 'tensor': 'lm_head.weight', 'status': 'validated-and-reshard-required'}
生产替换点：真实 Checkpoint 还需 safetensors、优化器/RNG、原子 COMPLETE 标记、对象存储 ETag、分布式并行布局、流式重分片和磁盘容量门禁。


## 回归测试：最后只保护顺序、完整性与结构兼容

In [7]:
assert not naive_equal and np.array_equal(safe_rebuilt, tensor)  # 验证文件名顺序失败而 Manifest 范围恢复逐值正确。
assert [item["file"] for item in ordered_specs] == ["shard-1.bin", "shard-2.bin", "shard-10.bin"]  # 验证逻辑范围顺序。
assert corrupt_tensor is None and corrupt_errors == ["hash:shard-2.bin"]  # 验证单字节损坏被精确定位。
assert wrong_tensor is None and config_errors == ["model-config-mismatch"]  # 验证隐藏宽度不兼容在加载前拒绝。
assert manifest["tokenizer_hash"] == "tok-45aa" and manifest["training_world_size"] == 3  # 验证恢复合同包含 Tokenizer 和保存并行规模。
print("回归测试通过：字典序反例、范围重建、单字节哈希、模型兼容和元数据合同均成立。")  # 用少量断言总结 Checkpoint 合同。

回归测试通过：字典序反例、范围重建、单字节哈希、模型兼容和元数据合同均成立。
